In [1]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

2.21.0


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2026-11-10 02:28:49--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.


HTTP request sent, awaiting response... 

200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv.2’

train-data.tsv.2      0%[                    ]       0  --.-KB/s               

train-data.tsv.2    100%[===================>] 349.84K  --.-KB/s    in 0.04s   

2026-11-10 02:28:49 (7.66 MB/s) - ‘train-data.tsv.2’ saved [358233/358233]



--2026-11-10 02:28:49--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv.2’

valid-data.tsv.2    100%[===================>] 115.99K  --.-KB/s    in 0.02s   

2026-11-10 02:28:49 (5.77 MB/s) - ‘valid-data.tsv.2’ saved [118774/118774]



In [3]:
# Load the data and turn labels into numbers: ham -> 0, spam -> 1.
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_labels = (train_df['label'] == 'spam').astype('float32').values
test_labels = (test_df['label'] == 'spam').astype('float32').values

# Text -> TF-IDF weights of single words and word pairs.
vectorizer = keras.layers.TextVectorization(max_tokens=10000, output_mode='tf_idf', ngrams=2)
vectorizer.adapt(train_df['message'].values)

train_df['label'].value_counts()

label
ham     3619
spam     560
Name: count, dtype: int64

In [4]:
tf.random.set_seed(0)
model = keras.Sequential([
    keras.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    keras.layers.Dropout(0.5),
    keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(tf.constant(train_df['message'].values), train_labels, epochs=30,
                    class_weight={0: 1.0, 1: 3.0},  # spam is rare, weigh it more
                    validation_data=(tf.constant(test_df['message'].values), test_labels), verbose=0)
print('Validation accuracy:', round(history.history['val_accuracy'][-1], 4))

Validation accuracy: 0.9878


In [5]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  probability = float(model.predict(tf.constant([pred_text]), verbose=0)[0][0])
  prediction = [probability, 'spam' if probability > 0.5 else 'ham']

  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

[0.01124750915914774, 'ham']


In [6]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!
